# Spatial autocorrelation

Build a spatial-weights matrix, measure global autocorrelation (Moran's I / Geary's C), then map local clusters
(LISA) and hotspots (Getis-Ord Gi*) on a synthetic lattice with a north-south gradient.

In [ ]:
import numpy as np
import geopandas as gpd
from shapely.geometry import box
from pyramids.feature import FeatureCollection
from geostatista import Weights, morans_i, gearys_c, local_morans, getis_ord_gi

n = 8
polys, value = [], []
for r in range(n):
    for c in range(n):
        polys.append(box(c, r, c + 1, r + 1))
        value.append(float(r))
gdf = gpd.GeoDataFrame({"v": value}, geometry=polys, crs="EPSG:32633")
tracts = FeatureCollection(gdf)
tracts.head()

## 1. Spatial weights — queen contiguity

In [ ]:
w = Weights.queen(tracts)
print("features:", w.n, " total neighbor links:", int(w.cardinalities.sum()))
print("cardinalities min/max:", w.cardinalities.min(), w.cardinalities.max())

## 2. Global autocorrelation

In [ ]:
mi = morans_i(tracts, "v", w, permutations=199, seed=0)
gc = gearys_c(tracts, "v", w, permutations=199, seed=0)
print(mi)
print(gc)

## 3. Local Moran (LISA) — per-feature cluster membership

In [ ]:
lisa = local_morans(tracts, "v", w, permutations=199, seed=0)
lisa["cluster"].value_counts()

## 4. Getis-Ord Gi* — hotspots

In [ ]:
hot = getis_ord_gi(tracts, "v", w, star=True)
hot[["v", "gi", "z", "hotspot"]].head(8)